# 04 — Feature Engineering & Spark Scaling


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from sklearn.preprocessing import StandardScaler
import mlflow
from neuro.bids import validate_bids
from neuro.features import build_feature_table, save_features_parquet, extract_roi_timeseries, get_schaefer_masker
from neuro.spark_utils import get_spark
from neuro.mlflow_utils import start_run

with start_run("04_feature_engineering"):
    report = validate_bids()
    runs = report["runs"]
    spark = get_spark("BladerunnerNeuro_Features")
    runs_pdf = runs[runs["bold_exists"]].copy()
    mlflow.log_param("n_runs", len(runs_pdf))
    print(f"Building features for {len(runs_pdf)} runs")


In [ ]:

masker = get_schaefer_masker(n_rois=100)

@F.udf(returnType=FloatType())
def roi_mean_scalar(path):
    ts = extract_roi_timeseries(path, masker)
    return float(ts.mean())

files_sdf = spark.createDataFrame(runs_pdf[["subject", "task", "run", "bold_path", "group_short"]])
files_sdf = files_sdf.withColumn("roi_global_mean", roi_mean_scalar(F.col("bold_path")))
spark_means = files_sdf.groupBy("group_short").avg("roi_global_mean").toPandas()
display(spark_means)
sns.barplot(data=spark_means, x="group_short", y="avg(roi_global_mean)", hue="group_short", palette="muted", legend=False)
plt.title("Spark-aggregated ROI global mean by group")
plt.show()


In [ ]:

feat_df = build_feature_table(runs_pdf, n_rois=100)
path = save_features_parquet(feat_df)
mlflow.log_artifact(str(path))
summary = feat_df[["ts_mean", "ts_std", "ts_entropy"]].describe()
display(summary)
sns.pairplot(feat_df, vars=["ts_mean", "ts_std", "ts_entropy"], hue="group_short", corner=True, plot_kws={"alpha": 0.6})
plt.show()
